# **Prediksi Potensi Produksi Padi di Indonesia Berdasarkan Kondisi Iklim Menggunakan Machine Learning**
----

SDGs Target: SDG 13 — Climate Action | SDG 2 — Zero Hunger.

Tim: JUJUR GW LUPA NAMANYA APA TP ITU POKOKNYA.

## 00. Dataset

In [1]:
!pip install kaggle

import os
os.makedirs('data/raw', exist_ok=True)

!kaggle datasets download -d greegtitan/indonesia-climate -p data/raw --unzip
!kaggle datasets download -d nurfianqodar/indonesian-rice-production-dataset -p data/raw --unzip

print("don")

Dataset URL: https://www.kaggle.com/datasets/greegtitan/indonesia-climate
License(s): copyright-authors




  0%|          | 0.00/7.15M [00:00<?, ?B/s]
 14%|█▍        | 1.00M/7.15M [00:00<00:05, 1.17MB/s]
 28%|██▊       | 2.00M/7.15M [00:01<00:02, 2.21MB/s]
 42%|████▏     | 3.00M/7.15M [00:01<00:01, 3.41MB/s]
 70%|██████▉   | 5.00M/7.15M [00:01<00:00, 6.07MB/s]
100%|██████████| 7.15M/7.15M [00:01<00:00, 5.29MB/s]


Dataset URL: https://www.kaggle.com/datasets/nurfianqodar/indonesian-rice-production-dataset
License(s): apache-2.0

don



  0%|          | 0.00/3.18k [00:00<?, ?B/s]
100%|██████████| 3.18k/3.18k [00:00<00:00, 4.42MB/s]


# 01. Data Collection & Exploration

In [5]:
!pip install pandas numpy matplotlib seaborn

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl (12.5 MB)
Using cached matplotlib-3.10.9-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl (232 kB)
Using cached cycler-0.12.1-py3-n

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [7]:
file_padi = '../data/raw/idrice.csv'
file_iklim = '../data/raw/climate_data.csv'

df_rice = pd.read_csv(file_padi)
df_climate = pd.read_csv(file_iklim)

display(df_rice.head())
display(df_climate.head())

,province,production,productivity,harvest_area,year
0,Aceh,1861567100,5649,329515.78,2018
1,Sumatera Utara,2108284720,5165,408176.45,2018
2,Sumatera Barat,1483076480,4737,313050.82,2018
3,Riau,266375530,3728,71448.08,2018
4,Jambi,383045740,4444,86202.68,2018


,date,Tn,Tx,Tavg,RH_avg,RR,ss,ff_x,ddd_x,ff_avg,ddd_car,station_id
0,01-01-2010,21.4,30.2,27.1,82.0,9.0,0.5,7.0,90.0,5.0,E,96001
1,02-01-2010,21.0,29.6,25.7,95.0,24.0,0.2,6.0,90.0,4.0,E,96001
2,03-01-2010,20.2,26.8,24.5,98.0,63.0,0.0,5.0,90.0,4.0,E,96001
3,04-01-2010,21.0,29.2,25.8,90.0,0.0,0.1,4.0,225.0,3.0,SW,96001
4,05-01-2010,21.2,30.0,26.7,90.0,2.0,0.4,NaN,NaN,NaN,NaN,96001


### Cek shape, dtypes, dan sample data

In [8]:
#check bentuk data
print("data padi:", df_rice.shape[0], "baris dan", df_rice.shape[1], "kolom.")
print("data iklim:", df_climate.shape[0], "baris dan", df_climate.shape[1], "kolom.")

data padi: 175 baris dan 5 kolom.
data iklim: 589265 baris dan 12 kolom.


In [14]:
#check data kosong
print("data kosong padi:")
print(df_rice.isnull().sum())
print("\ndata kosong iklim:")
print(df_climate.isnull().sum())

data kosong padi:
province        0
production      0
productivity    0
harvest_area    0
year            0
dtype: int64

data kosong iklim:
date               0
Tn             23383
Tx             37736
Tavg           45105
RH_avg         48182
RR            125384
ss             43721
ff_x           10214
ddd_x          13128
ff_avg         10127
ddd_car        13739
station_id         0
dtype: int64


In [15]:
#karena padi semuanya aman, tipe data iklim harus dicheck
df_climate.info()

<class 'pandas.DataFrame'>
RangeIndex: 589265 entries, 0 to 589264
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   date        589265 non-null  str    
 1   Tn          565882 non-null  float64
 2   Tx          551529 non-null  float64
 3   Tavg        544160 non-null  float64
 4   RH_avg      541083 non-null  float64
 5   RR          463881 non-null  float64
 6   ss          545544 non-null  float64
 7   ff_x        579051 non-null  float64
 8   ddd_x       576137 non-null  float64
 9   ff_avg      579138 non-null  float64
 10  ddd_car     575526 non-null  str    
 11  station_id  589265 non-null  int64  
dtypes: float64(9), int64(1), str(2)
memory usage: 53.9 MB


### Identifikasi duplikat & Cek distribusi nilai per kolom

In [16]:
print("data padi yang duplikat:", df_rice.duplicated().sum())
print("data iklim yang duplikat:", df_climate.duplicated().sum())

data padi yang duplikat: 0
data iklim yang duplikat: 0


In [17]:
display(df_rice.describe())
display(df_climate.describe())

,production,productivity,harvest_area,year
count,1.750000e+02,175.000000,1.750000e+02,175.000000
mean,3.172778e+09,4528.480000,6.123151e+05,2020.000000
std,9.393780e+09,873.900602,1.800893e+06,1.418272
min,5.069100e+05,2653.000000,1.794800e+02,2018.000000
25%,2.336388e+08,3913.000000,5.359704e+04,2019.000000
50%,5.334774e+08,4663.000000,1.258701e+05,2020.000000
75%,1.701110e+09,5156.000000,3.274245e+05,2021.000000
max,5.920053e+10,7276.000000,1.137793e+07,2022.000000


,Tn,Tx,Tavg,RH_avg,RR,ss,ff_x,ddd_x,ff_avg,station_id
count,565882.000000,551529.000000,544160.000000,541083.000000,463881.000000,545544.000000,579051.000000,576137.000000,579138.000000,589265.000000
mean,23.312111,31.528955,26.855475,82.489365,8.680760,5.083199,4.709601,188.488325,1.956680,96832.949230
std,2.280687,2.311659,1.939656,14.337669,17.928752,3.261586,2.612285,107.657452,1.803358,542.419161
min,0.000000,0.000000,0.000000,24.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,96001.000000
25%,23.000000,30.500000,26.200000,79.000000,0.000000,2.500000,3.000000,90.000000,1.000000,96293.000000
50%,24.000000,31.800000,27.200000,83.000000,1.000000,5.300000,4.000000,180.000000,2.000000,96797.000000
75%,25.000000,33.000000,28.000000,87.000000,9.300000,7.600000,6.000000,270.000000,3.000000,97240.000000
max,246.000000,334.000000,141.600000,7520.000000,1965.500000,705.000000,185.000000,931.000000,160.000000,97980.000000


**outlier data iklim**

| Data | Label | Data |
|---|---|---|
| Suhu Udara | Tn Tx (min) | 0.0C |
| Kelembapan | RH_avg (max) | 7520% |
| Curah Hujan | RR (min) | -1.0 |
| Durasi Sinar Matahari | ss (max) | 705 jam |

## 02. Data Preprocessing & Cleaning

In [18]:
print("jumlah data iklim sebelum cleaning:", df_climate.shape[0])

df_climate = df_climate[df_climate['Tx'] <= 50]
df_climate = df_climate[df_climate['Tn'] >= 0]
df_climate = df_climate[(df_climate['RH_avg'] >= 0) & (df_climate['RH_avg'] <= 100)]
df_climate = df_climate[df_climate['RR'] >= 0]

print("jumlah data iklim setelah cleaning:", df_climate.shape[0])

jumlah data iklim sebelum cleaning: 589265
jumlah data iklim setelah cleaning: 406444


data iklim belum ada belum ada province dan tahun karena hanya punya date dan station

In [19]:
df_station = pd.read_csv('data/raw/station_detail.csv')
df_province = pd.read_csv('data/raw/province_detail.csv')

display(df_station.head())
display(df_province.head())

,station_id,station_name,region_name,latitude,longitude,region_id,province_id
0,96001,Stasiun Meteorologi Maimun Saleh,Kota Sabang,5.87655,95.33785,20,1
1,96003,Balai Besar Meteorologi Klimatologi dan Geofi...,Kab. Badung,-8.73810,115.17860,272,17
2,96004,Balai Besar Meteorologi Klimatologi dan Geofis...,Kota Makassar,-5.14283,119.45227,412,26
3,96009,Stasiun Meteorologi Malikussaleh,Kab. Aceh Utara,5.22869,96.94749,8,1
4,96011,Stasiun Meteorologi Sultan Iskandar Muda,Kab. Aceh Besar,5.52244,95.41700,6,1


,province_id,province_name
0,1,Nanggroe Aceh Darussalam
1,2,Sumatera Utara
2,3,Sumatera Barat
3,4,Riau
4,5,Jambi


combine (iklim + station) + province. 
extract year

In [22]:
df_climate_st = pd.merge(df_climate, df_station, on='station_id', how='inner')
df_climate_full = pd.merge(df_climate_st, df_province, on='province_id', how='inner')
df_climate_full['year'] = pd.to_datetime(df_climate_full['date'], format='mixed', dayfirst=True).dt.year
df_climate_full.rename(columns={'province_name': 'province'}, inplace=True)

#data baru untuk iklim
kolom_penting = ['date', 'year', 'province', 'Tn', 'Tx', 'Tavg', 'RH_avg', 'RR']
display(df_climate_full[kolom_penting].head())

,date,year,province,Tn,Tx,Tavg,RH_avg,RR
0,01-01-2010,2010,Nanggroe Aceh Darussalam,21.4,30.2,27.1,82.0,9.0
1,02-01-2010,2010,Nanggroe Aceh Darussalam,21.0,29.6,25.7,95.0,24.0
2,03-01-2010,2010,Nanggroe Aceh Darussalam,20.2,26.8,24.5,98.0,63.0
3,04-01-2010,2010,Nanggroe Aceh Darussalam,21.0,29.2,25.8,90.0,0.0
4,05-01-2010,2010,Nanggroe Aceh Darussalam,21.2,30.0,26.7,90.0,2.0


In [24]:
#karena ini masih perhari jadi harus diganti jadi pertahun dulu
df_climate_full['province'] = df_climate_full['province'].str.lower().str.strip()
df_rice['province'] = df_rice['province'].str.lower().str.strip()

climate_yearly = df_climate_full.groupby(['province', 'year']).agg({
    'Tavg': 'mean',     #rata-rata suhu
    'RH_avg': 'mean',   #rata-rata kelembapan
    'RR': 'sum'         #total curah hujan
}).reset_index()

climate_yearly.rename(columns={
    'Tavg': 'avg_temperature',
    'RH_avg': 'avg_humidity',
    'RR': 'total_rainfall'
}, inplace=True)

display(climate_yearly.head())

,province,year,avg_temperature,avg_humidity,total_rainfall
0,bali,2010,27.445708,82.726037,10617.6
1,bali,2011,26.661805,81.023827,7621.2
2,bali,2012,26.835754,81.302326,5589.1
3,bali,2013,27.578533,79.656530,3378.0
4,bali,2014,27.440179,78.614796,3806.0


In [25]:
df_merged = pd.merge(climate_yearly, df_rice, on=['province', 'year'], how='inner')
print("total baris:", df_merged.shape[0])
print("total kolom:", df_merged.shape[1])

display(df_merged.head())
print("check data kosong:", df_merged.isnull().sum())

total baris: 93
total kolom: 8


,province,year,avg_temperature,avg_humidity,total_rainfall,production,productivity,harvest_area
0,bali,2018,27.287625,79.916388,1236.7,667069060,6011,110978.37
1,bali,2019,27.310903,79.355140,954.8,579320530,6078,95319.34
2,bali,2020,27.774803,80.787402,1227.1,532168450,5849,90980.69
3,banten,2018,27.553233,80.399469,7127.3,1687783300,4894,344836.06
4,banten,2019,27.854355,78.435484,6447.7,1470503350,4842,303731.80


check data kosong: province           0
year               0
avg_temperature    0
avg_humidity       0
total_rainfall     0
production         0
productivity       0
harvest_area       0
dtype: int64


In [26]:
os.makedirs('data/processed', exist_ok=True)
df_merged.to_csv('data/processed/df_merged_clean.csv', index=False)

print("don")

don


## 03. Feature Engineering